# 3. Obesity Risk Prediction

**Goal:** Predict binary obesity risk using behavioral and lifestyle features only.

`Height`, `Weight`, and `BMI` are intentionally excluded to avoid target leakage.

In [ ]:
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score, train_test_split

ARTIFACTS_DIR = '../artifacts'

In [ ]:
prediction_frame = pd.read_csv(f'{ARTIFACTS_DIR}/processed_data.csv')
CATEGORICAL_FEATURES = ['Gender', 'CALC', 'FAVC', 'SCC', 'SMOKE', 'family_history_with_overweight', 'CAEC', 'MTRANS']
CONTINUOUS_FEATURES = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
FEATURES = CATEGORICAL_FEATURES + CONTINUOUS_FEATURES
obese_labels = ['Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III']
prediction_frame['is_obese'] = prediction_frame['NObeyesdad'].isin(obese_labels).astype(int)
print(prediction_frame['is_obese'].value_counts())
print(f"Obesity rate: {prediction_frame['is_obese'].mean():.2%}")

## Train/test split

In [ ]:
X = prediction_frame[FEATURES]
y = prediction_frame['is_obese']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train rows: {len(X_train)} | Test rows: {len(X_test)}')

## Train logistic regression

In [ ]:
classifier = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
classifier.fit(X_train, y_train)
cv_auc_scores = cross_val_score(classifier, X_train, y_train, cv=5, scoring='roc_auc')
print(f'5-fold CV ROC AUC: {cv_auc_scores.mean():.4f} +/- {cv_auc_scores.std():.4f}')

## Evaluate the classifier

In [ ]:
predicted_labels = classifier.predict(X_test)
predicted_probabilities = classifier.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, predicted_labels)
roc_auc = roc_auc_score(y_test, predicted_probabilities)
print(f'Accuracy: {accuracy:.4f}')
print(f'ROC AUC: {roc_auc:.4f}')
print()
print(classification_report(y_test, predicted_labels, target_names=['Not Obese', 'Obese']))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
matrix = confusion_matrix(y_test, predicted_labels)
ConfusionMatrixDisplay(matrix, display_labels=['Not Obese', 'Obese']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix')
false_positive_rate, true_positive_rate, _ = roc_curve(y_test, predicted_probabilities)
axes[1].plot(false_positive_rate, true_positive_rate, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

## Feature importance via model coefficients

In [ ]:
coefficient_frame = pd.DataFrame({'Feature': FEATURES, 'Coefficient': classifier.coef_[0]})
coefficient_frame = coefficient_frame.sort_values('Coefficient', key=abs, ascending=False)
plt.figure(figsize=(10, 5))
colors = ['#e74c3c' if value > 0 else '#3498db' for value in coefficient_frame['Coefficient']]
sns.barplot(x='Coefficient', y='Feature', data=coefficient_frame, palette=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Coefficients')
plt.tight_layout()
plt.show()
coefficient_frame

## Save the trained model

In [ ]:
with open(f'{ARTIFACTS_DIR}/prediction_model.pkl', 'wb') as file:
    pickle.dump({'model': classifier, 'features': FEATURES, 'obese_labels': obese_labels}, file)
print('Saved artifacts/prediction_model.pkl')

## Inference helper

In [ ]:
def predict_obesity_risk(user_input: dict) -> dict:
    with open(f'{ARTIFACTS_DIR}/encoders.pkl', 'rb') as file:
        encoders = pickle.load(file)
    with open(f'{ARTIFACTS_DIR}/scaler.pkl', 'rb') as file:
        scaler = pickle.load(file)
    with open(f'{ARTIFACTS_DIR}/prediction_model.pkl', 'rb') as file:
        saved_model = pickle.load(file)

    row = dict(user_input)
    for column, mapping in encoders['binary'].items():
        if column in row:
            row[column] = mapping[row[column]]
    for column, order in encoders['ordinal'].items():
        if column in row:
            row[column] = {value: index for index, value in enumerate(order)}[row[column]]

    categorical = ['Gender', 'CALC', 'FAVC', 'SCC', 'SMOKE', 'family_history_with_overweight', 'CAEC', 'MTRANS']
    continuous = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
    scaled_continuous = scaler.transform([[row[column] for column in continuous]])[0]
    transformed = {column: scaled_continuous[index] for index, column in enumerate(continuous)}
    transformed.update({column: row[column] for column in categorical})

    feature_frame = pd.DataFrame([transformed])[saved_model['features']]
    probability = saved_model['model'].predict_proba(feature_frame)[0][1]
    if probability < 0.30:
        risk_label = 'Low risk'
    elif probability < 0.60:
        risk_label = 'Moderate risk'
    else:
        risk_label = 'High risk'
    return {'risk_score': round(float(probability), 4), 'label': risk_label}

example_input = {
    'Age': 28, 'Gender': 'Male', 'CALC': 'Sometimes', 'FAVC': 'yes',
    'FCVC': 1, 'NCP': 3, 'SCC': 'yes', 'SMOKE': 'no', 'CH2O': 1,
    'family_history_with_overweight': 'yes', 'FAF': 0, 'TUE': 2,
    'CAEC': 'Frequently', 'MTRANS': 'Automobile'
}
print(predict_obesity_risk(example_input))